<a href="https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import os
import subprocess

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [13]:
!pip -q install duckdb huggingface_hub pandas pyarrow

In [14]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="README.md",
    token=HF_TOKEN
)

print("Connected successfully!")
print(file_path)

Connected successfully!
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/README.md


In [15]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [16]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [17]:
import duckdb

con = duckdb.connect()

df = con.sql(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the daily performance of one content page (content_hash_id) for one client (client_hash_id). For this assignment, I use the March 2026 data (month = 2026-03). My goal is to rank content pages that may benefit from a content refresh using observable performance metrics.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows in dataset:", len(df))

print("\nDate range:")
print(df["report_date"].min(), "to", df["report_date"].max())

print("\nUnique clients:", df["client_hash_id"].nunique())

print("Unique content pages:", df["content_hash_id"].nunique())

Rows in dataset: 5

Date range:
2026-03-01 00:00:00 to 2026-03-01 00:00:00

Unique clients: 1
Unique content pages: 5


## 2. Fields: feature / label / context / excluded

**Features**
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- gsc_data_available

These are observable values available before making a content refresh decision.

**Label / Proxy**
The model will produce a priority score that ranks pages for content refresh rather than predicting a direct future outcome.

**Context**
- report_date
- client_hash_id
- content_hash_id
- month

These fields identify the page, client, and reporting period but are not used directly as predictive features.

**Excluded**
GA4-only fields are excluded because many rows do not have GA4 data available, which could introduce unnecessary missing values and reduce consistency.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "client_has_gsc",
    "gsc_data_available"
]

print("Selected Features:")
for col in feature_columns:
    print("-", col)

print("\nContext Fields:")
print(["report_date", "client_hash_id", "content_hash_id", "month"])

Selected Features:
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- gsc_data_available

Context Fields:
['report_date', 'client_hash_id', 'content_hash_id', 'month']


## 3. Verify it with queries

The following queries verify my data contract by checking:
1. The dataset grain (one row per content page per day).
2. The number of rows and reporting date range.
3. Data availability for Google Search Console fields.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

con = duckdb.connect()
con.register("performance", df)

# Query 1: Dataset size and date range
print("=== Query 1: Row count and Date Range ===")
print(con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM performance
""").df())

# Query 2: Verify grain
print("\n=== Query 2: Grain Check ===")
print(con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id || report_date::VARCHAR) AS unique_content_day
FROM performance
""").df())

# Query 3: GSC availability
print("\n=== Query 3: GSC Data Availability ===")
print(con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available
FROM performance
""").df())

=== Query 1: Row count and Date Range ===
   total_rows start_date   end_date
0           5 2026-03-01 2026-03-01

=== Query 2: Grain Check ===
   total_rows  unique_content_day
0           5                   5

=== Query 3: GSC Data Availability ===
   total_rows  gsc_available
0           5            5.0


## 4. Data limits

This dataset contains observed website performance metrics, but it cannot explain the true cause of changes in traffic or rankings. External factors such as Google algorithm updates, seasonality, competitor actions, or marketing campaigns are not fully captured. Therefore, this project should be used as decision-support rather than proof of cause-and-effect.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Limitation:")
print("- This dataset contains observed performance data only.")
print("- It cannot prove why traffic or rankings changed.")
print("- The model provides decision-support, not causal conclusions.")

Limitation:
- This dataset contains observed performance data only.
- It cannot prove why traffic or rankings changed.
- The model provides decision-support, not causal conclusions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.